[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-04b-context-engineering-as-cost-engineering.ipynb)

# Advanced Discussion 4 (continued) — caching, shaping, compaction and pinning

Cost of a 30-turn agent run under four context policies with prefix caching, tool-result truncation savings, and what each policy forgets.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in seconds. Prices are placeholders.

In [ ]:
%pip install -q numpy

In [ ]:
import numpy as np, math
IN, OUT = 3.00, 15.00; READ, WRITE = 0.10, 1.25          # PLACEHOLDER price ($/M tokens) and cache multipliers: use your provider's current numbers
PREFIX, T = 6000, 30                                       # fixed system prompt + tools + policy; turns per run
rs = np.random.RandomState(0)

## 1. one run: growth of context under four policies, and what it costs with prefix caching

In [ ]:
tool_tokens = np.clip(rs.lognormal(math.log(600), 0.8, T).astype(int), 50, 6000); ASSIST = 120
def shape(n, limit=300): return n if n <= limit else limit + 20          # head + tail + an 'N tokens omitted' marker
def run(policy, hi=12000, lo=7000):
    ctx = PREFIX; cached = 0; total = 0.0; peak = 0; cached_tok = 0; fresh_tok = 0
    for t in range(T):
        new = int(shape(tool_tokens[t]) if policy in ("truncate", "truncate+big-compaction") else tool_tokens[t]) + ASSIST
        # this request's input = previous context (cache may cover part of it) + new tokens
        reused = min(cached, ctx)                                                    # cache covers an unchanged prefix
        fresh = ctx - reused + new
        total += (reused * IN * READ + fresh * IN * WRITE + ASSIST * OUT) / 1e6      # new tokens are written to the cache
        cached_tok += reused; fresh_tok += fresh
        ctx += new; peak = max(peak, ctx); cached = ctx                              # everything so far is now cached
        if policy == "compact-every-turn" and ctx > lo:
            ctx = lo; cached = PREFIX                                                # history rewritten: only the fixed prefix still matches
        if policy in ("big-compaction", "truncate+big-compaction") and ctx > hi:
            ctx = lo; cached = PREFIX
    return total, peak, cached_tok / (cached_tok + fresh_tok)
print("30-turn agent run; fixed prefix %d tokens; tool results average %d tokens (heavy tail, max %d)" % (PREFIX, tool_tokens.mean(), tool_tokens.max()))
print("policy                                cost      peak context   share of input read from cache")
for name in ("append-only", "truncate", "compact-every-turn", "big-compaction", "truncate+big-compaction"):
    c, p, h = run(name); print("%-36s $%.4f    %6d          %.0f%%" % (name, c, p, 100 * h))
nocache = 0.0; ctx = PREFIX
for t in range(T): ctx += int(tool_tokens[t]) + ASSIST; nocache += ((ctx) * IN + ASSIST * OUT) / 1e6
print("(no caching, append-only, for reference: $%.4f)" % nocache)

## 2. what a policy forgets: five constraints stated early, needed at the end

In [ ]:
def survive(policy, seed, n_facts=5, window=8000):
    r = np.random.RandomState(seed); when = np.sort(r.randint(0, 10, n_facts)); toks = np.clip(r.lognormal(math.log(600), 0.8, T).astype(int), 50, 6000) + ASSIST
    starts = np.concatenate([[0], np.cumsum(toks)])[:-1]; total = toks.sum()
    if policy == "keep last 8k tokens (FIFO)":
        return all(total - starts[w] <= window for w in when)                       # a fact survives only if it is still inside the window
    n_compactions = max(0, int((total + PREFIX) // 12000))
    if policy == "lossy summary at each compaction":
        return all(r.rand() < 0.85 ** n_compactions for _ in range(n_facts))       # each fact survives each rewrite with probability 0.85
    if policy == "pinned constraints + recent turns":
        return True                                                                # facts are copied into a pinned block that is never dropped
def rate(policy, trials=2000): return np.mean([survive(policy, s) for s in range(trials)])
print("\nshare of runs in which all 5 early constraints are still available at turn 30:")
for pol in ("keep last 8k tokens (FIFO)", "lossy summary at each compaction", "pinned constraints + recent turns"): print("  %-36s %.3f" % (pol, rate(pol)))
print("pinned block cost: 5 constraints x ~40 tokens = %d tokens per request (%.1f%% of the fixed prefix)" % (200, 100 * 200 / PREFIX))
print("probability a fact survives k lossy compactions at 0.85 each:", "  ".join("k=%d: %.2f" % (k, 0.85 ** k) for k in (1, 2, 4, 8)))

## 3. tool-result shaping: what the limit saves

In [ ]:
big = tool_tokens
for lim in (200, 300, 600, 1000):
    shaped = np.array([shape(n, lim) for n in big]); print("limit %4d tokens: mean result %.0f -> %.0f tokens (%.0f%% saved), %d of %d results truncated" % (lim, big.mean(), shaped.mean(), 100 * (1 - shaped.sum() / big.sum()), (big > lim).sum(), T))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.